# CoWork Enterprise Demo — Context Layer
## Notebook 01 — Build the Context Layer (Phase 1)

_Icons used throughout: 🛠️ Action  📌 Note  🔹 Info_

> ⚠️ _Generated from `build_notebooks.py` — edit the builder and re-run. Direct edits to this notebook are overwritten._

---

### What This Notebook Does:

1. 🛠️ Creates a **governed semantic view** — logical tables, relationships, dimensions, metrics, and verified queries — as the single source of truth for structured questions
2. 🛠️ Creates a **Cortex Search service** over the unstructured research notes (RAG)
3. 🛠️ **Validates** the semantic view returns correct numbers *before* any agent is built on it

---

### Why the Context Layer Matters:

🔹 The context layer is the **single biggest driver of answer quality**. Without it, an AI agent guesses at column meanings and joins. With it:

- The agent **knows** `AUM` means Assets Under Management, not a column to guess at
- The agent **knows** how to join CLIENTS → POSITIONS → TRADES
- **Verified queries** teach the agent the *correct* SQL for common questions
- Data-layer governance (grants, masking) flows through automatically

> **Documentation:** [Semantic Views](https://docs.snowflake.com/en/user-guide/views-semantic/overview) | [Cortex Search](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview)

---

### Estimated Time: **15–20 minutes**

### Prerequisites:
- Notebook 00 complete (`COWORK_ENTERPRISE_DEMO` with analytics tables loaded)

## Step 1: Set Context

🛠️ Point the session at the demo role, database, `SEMANTIC` schema, and warehouse before creating objects.

In [ ]:
%%sql -r set_context
USE ROLE COWORK_ENTERPRISE_DEMO_ADMIN;
USE DATABASE COWORK_ENTERPRISE_DEMO;
USE SCHEMA SEMANTIC;
USE WAREHOUSE COWORK_ENTERPRISE_DEMO_WH;


### 🔹 Two Ways to Build: SQL vs. the UI

We author the semantic view and search services in **SQL** here — full control over every attribute, relationship, and metric, and it's reproducible and version-controllable (this whole guide is generated from code). You can also build both in **Snowsight**:

| Object | UI path | What it gives you |
|---|---|---|
| Semantic view | AI & ML → Cortex Analyst → **Create with Autopilot** | Auto-proposes dimensions, metrics, and relationships from your tables, which you then refine |
| Cortex Search service | AI & ML → Cortex Search → **Create** | Guided wizard for source table, search column, and attributes |

🔹 **Autopilot** is great for a fast first draft; we use SQL so every component is explicit and you understand exactly what you're shipping.

## Step 2: Enable Fuzzy Matching on `CLIENT_NAME`

🛠️ `CLIENT_NAME` is a **high-cardinality identifier** column — users will type *"Acme"* when the stored value is *"Acme Capital Partners LLC"*. We create a **Cortex Search service over its distinct values** so Cortex Analyst can resolve those fuzzy references to the exact literal it needs in the `WHERE` clause. (Step 3 links this service to the `CLIENT_NAME` dimension.)

🔹 **Why this column and not others:** use a search service for high-cardinality identifier text (names, SKUs). For low-cardinality categoricals like `REGION` or `SECTOR` (a handful of values), sample values are better; and never point a search service at numeric/date columns or paragraph free text.

> **Documentation:** [Improve literal search for Cortex Analyst](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-analyst/cortex-analyst-search-integration)

In [ ]:
%%sql -r create_client_search
CREATE OR REPLACE CORTEX SEARCH SERVICE COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_CLIENT_SEARCH
  ON CLIENT_NAME
  WAREHOUSE = COWORK_ENTERPRISE_DEMO_WH
  TARGET_LAG = '1 hour'
  COMMENT = 'Fuzzy literal matching for client names, linked to the DEMO_SV CLIENT_NAME dimension'
AS (
  SELECT DISTINCT CLIENT_NAME FROM COWORK_ENTERPRISE_DEMO.ANALYTICS.CLIENTS
);


## Step 3: Create the Semantic View

🛠️ This defines the meaning on top of the raw tables:

- **Logical tables**: CLIENTS, POSITIONS, TRADES
- **Relationships**: how they join (on `CLIENT_ID`)
- **Dimensions**: client attributes, region, sector, time
- **Metrics**: AUM, portfolio value, unrealized PnL, trade volume
- **Verified queries**: worked examples that teach the agent correct SQL patterns

🔹 The `CLIENT_NAME` dimension is defined `WITH CORTEX SEARCH SERVICE ...`, wiring in the fuzzy matching from Step 2. A single Cortex Analyst call operates against **one** semantic view, so keep tightly-related tables together (as here) and split genuinely distinct domains into separate views.

In [ ]:
%%sql -r create_semantic_view
CREATE OR REPLACE SEMANTIC VIEW COWORK_ENTERPRISE_DEMO.SEMANTIC.DEMO_SV

  TABLES (
    clients AS COWORK_ENTERPRISE_DEMO.ANALYTICS.CLIENTS
      PRIMARY KEY (CLIENT_ID)
      COMMENT = 'Client master data including institutional and HNW accounts with AUM and risk profiles',

    positions AS COWORK_ENTERPRISE_DEMO.ANALYTICS.POSITIONS
      PRIMARY KEY (POSITION_ID)
      COMMENT = 'Current portfolio positions by client and symbol with market values and unrealized PnL',

    trades AS COWORK_ENTERPRISE_DEMO.ANALYTICS.TRADES
      PRIMARY KEY (TRADE_ID)
      COMMENT = 'Trade execution history including buy/sell orders with prices, quantities, and status'
  )

  RELATIONSHIPS (
    positions_to_clients AS
      positions(CLIENT_ID) REFERENCES clients,
    trades_to_clients AS
      trades(CLIENT_ID) REFERENCES clients
  )

  FACTS (
    clients.AUM AS AUM
      COMMENT = 'Assets Under Management in USD. Total value of assets the client has with Nexus Capital.',
    positions.POSITION_QUANTITY AS QUANTITY
      COMMENT = 'Number of shares held in the position',
    positions.AVG_COST AS AVG_COST
      COMMENT = 'Average cost basis per share for the position',
    positions.CURRENT_PRICE AS CURRENT_PRICE
      COMMENT = 'Current market price per share',
    positions.MARKET_VALUE AS MARKET_VALUE
      COMMENT = 'Current market value of the position (quantity * current_price)',
    positions.UNREALIZED_PNL AS UNREALIZED_PNL
      COMMENT = 'Unrealized profit or loss on the position (market_value - cost_basis). Positive = gain, negative = loss.',
    trades.TRADE_QUANTITY AS QUANTITY
      COMMENT = 'Number of shares in the trade order',
    trades.TRADE_PRICE AS PRICE
      COMMENT = 'Execution price per share for the trade'
  )

  DIMENSIONS (
    clients.CLIENT_NAME AS CLIENT_NAME
      COMMENT = 'Full name of the client account'
      WITH CORTEX SEARCH SERVICE COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_CLIENT_SEARCH,
    clients.ACCOUNT_TYPE AS ACCOUNT_TYPE
      COMMENT = 'Account classification: INSTITUTIONAL, HNW (High Net Worth), or RETAIL',
    clients.REGION AS REGION
      COMMENT = 'Geographic region: North America, EMEA, or APJ',
    clients.RISK_PROFILE AS RISK_PROFILE
      COMMENT = 'Investment risk tolerance: CONSERVATIVE, MODERATE, or AGGRESSIVE',
    clients.RELATIONSHIP_MANAGER AS RELATIONSHIP_MANAGER
      COMMENT = 'Name of the assigned relationship manager',
    clients.STATUS AS STATUS
      COMMENT = 'Client account status: ACTIVE or INACTIVE',
    clients.ONBOARDED_DATE AS ONBOARDED_DATE
      COMMENT = 'Date the client was onboarded',

    positions.SYMBOL AS SYMBOL
      COMMENT = 'Stock ticker symbol (e.g., AAPL, NVDA, MSFT)',
    positions.SECTOR AS SECTOR
      COMMENT = 'Market sector classification (Technology, Financials, Energy, Healthcare, etc.)',

    trades.TRADE_TYPE AS TRADE_TYPE
      COMMENT = 'Direction of the trade: BUY or SELL',
    trades.TRADE_STATUS AS STATUS
      COMMENT = 'Trade execution status: EXECUTED or SETTLED',
    trades.EXCHANGE AS EXCHANGE
      COMMENT = 'Exchange where trade was executed (NYSE, NASDAQ, OTC)',
    trades.TRADE_DATE AS TRADE_DATE
      COMMENT = 'Timestamp when the trade was executed',
    trades.TRADE_NOTES AS NOTES
      COMMENT = 'Trader notes explaining the rationale for the trade'
  )

  METRICS (
    clients.TOTAL_AUM AS SUM(clients.AUM)
      COMMENT = 'Total Assets Under Management across all clients in USD',

    clients.CLIENT_COUNT AS COUNT(DISTINCT CLIENT_ID)
      COMMENT = 'Number of distinct client accounts',

    positions.TOTAL_PORTFOLIO_VALUE AS SUM(positions.MARKET_VALUE)
      COMMENT = 'Total current market value across all positions in USD',

    positions.TOTAL_UNREALIZED_PNL AS SUM(positions.UNREALIZED_PNL)
      COMMENT = 'Total unrealized profit/loss across all positions. Positive = overall portfolio gains.',

    positions.POSITION_COUNT AS COUNT(DISTINCT POSITION_ID)
      COMMENT = 'Number of distinct portfolio positions',

    positions.AVG_POSITION_VALUE AS AVG(positions.MARKET_VALUE)
      COMMENT = 'Average market value per position',

    trades.TRADE_COUNT AS COUNT(DISTINCT TRADE_ID)
      COMMENT = 'Number of trade executions',

    trades.TOTAL_TRADE_VOLUME AS SUM(trades.TRADE_QUANTITY * trades.TRADE_PRICE)
      COMMENT = 'Total dollar volume of trades (quantity * price summed across all trades)'
  )

  COMMENT = 'Nexus Capital - Financial analytics semantic view covering clients, positions, and trades'

  AI_SQL_GENERATION '
    -- Business rules for SQL generation:
    -- 1. When asked about "top clients" or "biggest clients", rank by AUM unless otherwise specified.
    -- 2. When asked about portfolio performance, use UNREALIZED_PNL. Positive = gains.
    -- 3. Default time filter: if no date specified, include all available data.
    -- 4. "Active clients" means STATUS = ''ACTIVE''.
    -- 5. When asked about "recent trades", order by TRADE_DATE DESC and limit to last 7 days unless specified.
    -- 6. Trade volume = QUANTITY * PRICE. Always compute as dollar volume, not share count.
    -- 7. For region breakdowns, use the CLIENTS.REGION column (North America, EMEA, APJ).
    -- 8. When computing concentration, use MARKET_VALUE / total portfolio MARKET_VALUE.
  '

  AI_VERIFIED_QUERIES (
    top_clients_by_aum AS (
      QUESTION 'What are our top 5 clients by AUM?'
      SQL 'SELECT CLIENT_NAME, ACCOUNT_TYPE, REGION, AUM, RISK_PROFILE
      FROM COWORK_ENTERPRISE_DEMO.ANALYTICS.CLIENTS
      WHERE STATUS = ''ACTIVE''
      ORDER BY AUM DESC
      LIMIT 5'
    ),

    portfolio_value_by_sector AS (
      QUESTION 'What is the total portfolio value by sector?'
      SQL 'SELECT p.SECTOR, SUM(p.MARKET_VALUE) AS TOTAL_VALUE, SUM(p.UNREALIZED_PNL) AS TOTAL_PNL
      FROM COWORK_ENTERPRISE_DEMO.ANALYTICS.POSITIONS p
      GROUP BY p.SECTOR
      ORDER BY TOTAL_VALUE DESC'
    ),

    recent_large_buy_trades AS (
      QUESTION 'Show me recent buy trades over $1M in value'
      SQL 'SELECT c.CLIENT_NAME, t.SYMBOL, t.QUANTITY, t.PRICE, (t.QUANTITY * t.PRICE) AS TRADE_VALUE, t.TRADE_DATE, t.NOTES
      FROM COWORK_ENTERPRISE_DEMO.ANALYTICS.TRADES t
      JOIN COWORK_ENTERPRISE_DEMO.ANALYTICS.CLIENTS c ON t.CLIENT_ID = c.CLIENT_ID
      WHERE t.TRADE_TYPE = ''BUY'' AND (t.QUANTITY * t.PRICE) > 1000000
      ORDER BY t.TRADE_DATE DESC'
    ),

    clients_highest_unrealized_gains AS (
      QUESTION 'Which clients have the highest unrealized gains?'
      SQL 'SELECT c.CLIENT_NAME, c.ACCOUNT_TYPE, SUM(p.UNREALIZED_PNL) AS TOTAL_UNREALIZED_PNL, SUM(p.MARKET_VALUE) AS TOTAL_PORTFOLIO_VALUE
      FROM COWORK_ENTERPRISE_DEMO.ANALYTICS.CLIENTS c
      JOIN COWORK_ENTERPRISE_DEMO.ANALYTICS.POSITIONS p ON c.CLIENT_ID = p.CLIENT_ID
      GROUP BY c.CLIENT_NAME, c.ACCOUNT_TYPE
      ORDER BY TOTAL_UNREALIZED_PNL DESC'
    ),

    aum_breakdown_by_region AS (
      QUESTION 'What is our AUM breakdown by region?'
      SQL 'SELECT REGION, COUNT(*) AS CLIENT_COUNT, SUM(AUM) AS TOTAL_AUM, AVG(AUM) AS AVG_AUM
      FROM COWORK_ENTERPRISE_DEMO.ANALYTICS.CLIENTS
      WHERE STATUS = ''ACTIVE''
      GROUP BY REGION
      ORDER BY TOTAL_AUM DESC'
    ),

    tech_sector_exposure AS (
      QUESTION 'Show me the technology sector exposure across all clients'
      SQL 'SELECT c.CLIENT_NAME, p.SYMBOL, p.QUANTITY, p.MARKET_VALUE, p.UNREALIZED_PNL
      FROM COWORK_ENTERPRISE_DEMO.ANALYTICS.POSITIONS p
      JOIN COWORK_ENTERPRISE_DEMO.ANALYTICS.CLIENTS c ON p.CLIENT_ID = c.CLIENT_ID
      WHERE p.SECTOR = ''Technology''
      ORDER BY p.MARKET_VALUE DESC'
    )
  );

SELECT 'Semantic view DEMO_SV created successfully' AS STATUS;


### 🔹 Best Practices: What Makes a Semantic View Accurate

Snowflake's [semantic-view best practices](https://www.snowflake.com/en/developers/guides/best-practices-semantic-views-cortex-analyst/) distill to a few high-leverage rules — the demo view above follows them:

- **Descriptions are the #1 element (required, not optional).** LLMs can't infer proprietary terms, so every table and column needs a clear, business-language description that spells out abbreviations (e.g. *"CSAT: customer satisfaction, 1–5; also called KPI_1 in legacy systems"*), not just `"score"`.
- **Relationships must be explicit.** Cortex Analyst **will not join** tables unless the relationship is defined. (Many-to-many isn't directly supported — bridge it with a shared dimension table.)
- **Define metrics and filters wherever possible.** They're commonly underused but are critical for accuracy — a metric like `total_revenue = SUM(unit_price*quantity)` stops the model from re-deriving business math each time.
- **Verified queries are the biggest accuracy lever.** Start with **10–20** covering your most common questions and grow from real usage. They only load when a question is *semantically similar*, so they rarely add token cost — and they seed better metric/filter/description suggestions.
- **Think like the end user.** Model by business domain and terminology, not table structure. Include only columns users will actually ask about.
- **Synonyms are no longer recommended** with frontier models — reserve them for genuinely unique or industry-specific terms the model won't know.

### 🔹 Best Practices: Custom Instructions (two distinct types)

The most common custom-instruction mistake is mixing up the two types:

| Type | Applies to | Use for |
|---|---|---|
| **`sql_generation`** | Every generated query | Business SQL logic, default filters, fiscal-year definitions, data quirks (e.g. *"fiscal year starts Feb 1"*, *"always use net_revenue"*) |
| **`question_categorization`** | Question understanding | When to **reject** questions (e.g. salaries, profanity) or **ask a clarifying question** (e.g. define *"active users"*) |

📌 Be **specific** (*"if no date filter is provided, default to the last year"* — not *"filter by last year"*) and **always test after adding instructions** (the Analyst playground is ideal).

### 📐 Design Note: One Semantic View vs. Many

`DEMO_SV` covers one domain (clients, positions, trades — all related, all joined on `CLIENT_ID`). In real deployments, *"how many semantic views?"* comes up fast.

**Key constraint:** a single Cortex Analyst call operates against **one** semantic view. The agent can't generate a SQL JOIN across two separate SVs — but it *can* call them sequentially and synthesize the results.

| Situation | Recommendation |
|---|---|
| Tables naturally join within one domain | One SV — keep it together |
| ~100+ columns | Split by domain; accuracy degrades above the soft limit |
| Distinct domains (Sales, HR, Finance) | Separate SVs + one Cortex Analyst tool per SV on the agent |
| Cross-domain JOIN needed at query time | Unified SV with RELATIONSHIPS (or Composable SVs) |

🔹 **`DEMO_SV` is the right scope:** CLIENTS / POSITIONS / TRADES are tightly related and questions span all three. A separate HR or Compliance domain would warrant its own SV, with the agent given one Analyst tool per SV. **Field note:** the ~100-column soft limit is a useful forcing function — it pushes domain decomposition that also makes the agent more accurate.

### 🧠 Going Deeper: Ontology & Knowledge Graphs

A semantic view defines meaning for **one domain**. When an enterprise has many interconnected domains and the AI must reason *across* them, **Ontology on Snowflake** extends the idea with a formal ontology (shared vocabulary + business logic across domains), a **knowledge graph** (entity relationships spanning systems), and agent-based reasoning for multi-hop questions.

📌 Relevant when customers say *"our AI gives inconsistent answers depending on which team asks"* or *"we need explainable AI that can trace its reasoning"* (often evaluated against Palantir / Neo4j). **Not required for this guide** — but a pattern worth knowing.

> **Reference:** [Ontology on Snowflake (GitHub)](https://github.com/Snowflake-Labs/ontology-on-snowflake) · [Ontology Stack Builder CoCo skill](https://github.com/Snowflake-Labs/cortex-code-skills/tree/main/skills/ontology-stack-builder)

## Step 4: Create the Cortex Search Service (research notes)

🛠️ This is a **second, different** search service — over the unstructured research notes for **RAG** (Retrieval-Augmented Generation). It lets the agent answer qualitative questions (analyst opinions, risk assessments, compliance reports) that no SQL query could.

🔹 Note the contrast with Step 2: **Step 2** indexes a structured *identifier column* for fuzzy literal matching; **this** indexes *document bodies* for semantic retrieval. Both are Cortex Search, used for different jobs.

In [ ]:
%%sql -r create_search_service
CREATE OR REPLACE CORTEX SEARCH SERVICE COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_SEARCH
  ON CONTENT
  ATTRIBUTES TITLE, AUTHOR, CATEGORY, REGION, SYMBOLS_MENTIONED
  WAREHOUSE = COWORK_ENTERPRISE_DEMO_WH
  TARGET_LAG = '1 hour'
  COMMENT = 'Search service over Nexus Capital research notes, market commentary, and compliance reports'
AS (
  SELECT
    NOTE_ID,
    TITLE,
    CONTENT,
    AUTHOR,
    CATEGORY,
    REGION,
    SYMBOLS_MENTIONED,
    PUBLISHED_DATE
  FROM COWORK_ENTERPRISE_DEMO.ANALYTICS.RESEARCH_NOTES
);

SELECT 'Cortex Search service DEMO_SEARCH created' AS STATUS;


## Step 5: Validate (the pre-agent gate)

📌 **Test the semantic view directly with `SEMANTIC_VIEW()` before building the agent.** If the view returns wrong numbers, the agent will too — catch it here, not in front of a business user.

In [ ]:
%%sql -r test_sv_clients
SELECT * FROM SEMANTIC_VIEW(
  COWORK_ENTERPRISE_DEMO.SEMANTIC.DEMO_SV
  DIMENSIONS CLIENTS.CLIENT_NAME
  METRICS POSITIONS.TOTAL_PORTFOLIO_VALUE
)
ORDER BY TOTAL_PORTFOLIO_VALUE DESC
LIMIT 5;


In [ ]:
%%sql -r test_sv_region
SELECT * FROM SEMANTIC_VIEW(
  COWORK_ENTERPRISE_DEMO.SEMANTIC.DEMO_SV
  DIMENSIONS CLIENTS.REGION
  METRICS CLIENTS.TOTAL_AUM
)
ORDER BY TOTAL_AUM DESC;


In [ ]:
%%sql -r show_search
SHOW CORTEX SEARCH SERVICES IN SCHEMA COWORK_ENTERPRISE_DEMO.AGENTS;


## 🧠 Common Pitfalls & Production Checklist

From the field, the semantic-view mistakes that most hurt accuracy:

| Pitfall | Fix |
|---|---|
| **Undefined scope** ("just one more table") | Set crisp success criteria and boundaries upfront |
| **Starting too big** (whole warehouse day one) | Start with **5–10 tables**, one domain, one use case |
| **Skipping verified queries** | Add **10–20** covering frequent questions from the start |
| **Wrong first use case** (Finance/Legal, ~100% bar) | Begin with lower-stakes domains (Sales/Marketing) |

**Production-ready checklist for a semantic view:**

- ✅ Every table and column has a clear business description; abbreviations explained
- ✅ 10–20 verified queries covering common questions
- ✅ Metrics for reusable calculations; filters for common conditions
- ✅ `sql_generation` custom instructions for business-specific logic
- ✅ Cortex Search on high-cardinality identifier columns
- ✅ An evaluation set + measured SQL accuracy before deployment (see Notebook 05)
- ✅ A weekly loop to add verified queries from real usage (see Notebook 07)

> **Reference:** [Best Practices for Semantic Views](https://www.snowflake.com/en/developers/guides/best-practices-semantic-views-cortex-analyst/) · [Evaluation tool](https://github.com/Snowflake-Labs/semantic-model-generator/)

## ✅ Notebook 01 Complete

### What You Just Built:
- Semantic view `COWORK_ENTERPRISE_DEMO.SEMANTIC.DEMO_SV` — logical tables, relationships, dimensions, metrics, verified queries
- Cortex Search service `COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_SEARCH` over the research notes
- Cortex Search service `COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_CLIENT_SEARCH` on `CLIENT_NAME` for fuzzy literal matching
- Validation that the view returns correct numbers

---

### Key Points:
- The semantic layer gives the agent *understanding*, not just *access*.
- Verified queries are like unit tests for your AI — they prove the agent generates correct SQL for known questions.
- Always validate the view directly before layering an agent on top.

---

### Next:

Continue to **Notebook 02 — Govern AI Access**: create a CoWork-only consumer role, grant the four layers of access, and prove masking policies apply to agent output automatically.